In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import SGDRegressor
from sklearn.linear_model import HuberRegressor
from tqdm import tqdm
import statsmodels.api as sm
from sklearn.metrics import r2_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset
import torch.nn.functional as F
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

# 老师服务器内存不够，这个实在跑不完ORZ😢

In [ ]:
with open("data/characteristics_with_ind_interaction_return.pkl", "rb") as f:
    data = pickle.load(f)

In [ ]:
memory_info = os.popen('free -h').read()
print(memory_info)

In [ ]:
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')

In [ ]:
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

pred_y_list = []
true_y_list = []

# 遍历测试年份
for year in tqdm(range(start_year_test, end_year_test + 1), desc='Processing', unit='item'):
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year - 13, month=12, day=31)
    start_date_val = pd.Timestamp(year=year - 12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year - 1, month=12, day=31)
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)

    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]

    X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_train = train_data['RET'].values
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_test = test_data['RET'].values
    # 初始化 GBRT 模型
    gbrt = GradientBoostingRegressor(n_estimators=100, learning_rate=0.01, random_state=42)

    # 训练模型
    gbrt.fit(X_train, y_train)

    # 预测
    y_test_pred = gbrt.predict(X_test)

    # 记录预测值和真实值
    pred_y_list.extend(y_test_pred)
    true_y_list.extend(y_test)
    print(r2_score(y_test,y_test_pred))
    tmp=pd.DataFrame({'pred':[i[0] for i in np.array(pred_y_list)],'true':[i[0] for i in np.array(true_y_list)]})
    tmp.to_csv(f'six/{year}.csv')
    gc.collect()
# 计算 R^2 分数
r2_score = r2_score(true_y_list,pred_y_list)
print('R^2: {:.2f}%'.format(r2_score * 100))